In [1]:
!pip install pandas
!pip install matplotlib
!pip install numpy

In [2]:
import pandas as pd
import numpy as np

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [5]:
engine_df = pd.read_csv("raw_data/engine.csv")
print("Total rows: ", len(engine_df))
engine_df.head()

Total rows:  4738


,CODE,MFR,MODEL,TYPE,HORSEPOWER,THRUST,Unnamed: 6
0,0,NONE,NONE,0,0,0,NaN
1,401,A.C.E.,HIDR MARK III,1,95,0,NaN
2,402,A.C.E.,UPRI MARK III,1,100,0,NaN
3,450,AEROMOMENT,AM13 SERIES,8,100,0,NaN
4,452,AEROMOMENT,AM15 SERIES,8,117,0,NaN


In [6]:
engine_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4738 entries, 0 to 4737
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   CODE        4738 non-null   int64  
 1   MFR         4738 non-null   str    
 2   MODEL       4738 non-null   str    
 3   TYPE        4738 non-null   str    
 4   HORSEPOWER  4738 non-null   int64  
 5   THRUST      4738 non-null   int64  
 6   Unnamed: 6  0 non-null      float64
dtypes: float64(1), int64(3), str(3)
memory usage: 259.2 KB


In [7]:
engine_df.columns = engine_df.columns.str.strip()

# 2. Identify object columns, then exclude 'CODE'
df_obj_cols = engine_df.select_dtypes(['object']).columns

# 3. Apply stripping only to the filtered list
engine_df[df_obj_cols] = engine_df[df_obj_cols].apply(lambda x: x.astype(str).str.strip())

# 4. Drop the ghost column if it exists
if 'Unnamed: 6' in engine_df.columns:
    engine_df = engine_df.drop(columns=['Unnamed: 6'])

engine_df.head()

/var/folders/r5/7sdqfmvs4p987n2g0n_nng840000gn/T/ipykernel_47839/2766345116.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_obj_cols = engine_df.select_dtypes(['object']).columns


,CODE,MFR,MODEL,TYPE,HORSEPOWER,THRUST
0,0,NONE,NONE,0,0,0
1,401,A.C.E.,HIDR MARK III,1,95,0
2,402,A.C.E.,UPRI MARK III,1,100,0
3,450,AEROMOMENT,AM13 SERIES,8,100,0
4,452,AEROMOMENT,AM15 SERIES,8,117,0


In [8]:
cols_to_fix = engine_df.columns

# Replace empty/whitespace strings with NaN only in those columns
engine_df[cols_to_fix] = engine_df[cols_to_fix].replace(r'^\s*$', np.nan, regex=True)

In [9]:
engine_df

,CODE,MFR,MODEL,TYPE,HORSEPOWER,THRUST
0,0,NONE,NONE,0,0,0
1,401,A.C.E.,HIDR MARK III,1,95,0
2,402,A.C.E.,UPRI MARK III,1,100,0
3,450,AEROMOMENT,AM13 SERIES,8,100,0
4,452,AEROMOMENT,AM15 SERIES,8,117,0
...,...,...,...,...,...,...
4733,83358,KDE,7215XF,10,135,0
4734,83359,KDE,HACKER,10,495,0
4735,83360,KDE,10218XF-105,10,140,0
4736,99222,MAGICALL,MAGIDRIVE 75,10,350,0


## Cleaning `TYPE` column

In [12]:
engine_df['TYPE'].value_counts()

TYPE
1     2398
2      750
5      712
3      397
4      191
8      172
7       72
10      39
11       4
0        1
9        1
Name: count, dtype: int64

In [14]:
engine_type_map = {
    '0': 'None',
    '1': 'Reciprocating',
    '2': 'Turbo-prop',
    '3': 'Turbo-shaft',
    '4': 'Turbo-jet',
    '5': 'Turbo-fan',
    '6': 'Ramjet',
    '7': '2 Cycle',
    '8': '4 Cycle',
    '9': 'Unknown',
    '10': 'Electric',
    '11': 'Rotary'
}

engine_df['TYPE'] = engine_df['TYPE'].astype(str).map(engine_type_map)

In [15]:
engine_df['TYPE'].value_counts()

TYPE
Reciprocating    2398
Turbo-prop        750
Turbo-fan         712
Turbo-shaft       397
Turbo-jet         191
4 Cycle           172
2 Cycle            72
Electric           39
Rotary              4
None                1
Unknown             1
Name: count, dtype: int64

## Cleaning `HORSEPOWER` column
- Set anything with 0 horsepower as `NaN`

In [26]:
engine_df['HORSEPOWER'].value_counts(dropna=False)

HORSEPOWER
<NA>    854
180     207
575     116
300     101
260     100
       ... 
123       1
212       1
27        1
148       1
247       1
Name: count, Length: 461, dtype: Int64

In [22]:
engine_df['HORSEPOWER'] = engine_df['HORSEPOWER'].astype('Int64')
engine_df['HORSEPOWER'] = engine_df['HORSEPOWER'].replace(0, np.nan)

In [27]:
engine_df['HORSEPOWER'].value_counts(dropna=False)

HORSEPOWER
<NA>    854
180     207
575     116
300     101
260     100
       ... 
123       1
212       1
27        1
148       1
247       1
Name: count, Length: 461, dtype: Int64

## Cleaning `THRUST` column
- Set anything with 0lb thrust to `NaN`

In [30]:
engine_df['THRUST'].value_counts(dropna=False)

THRUST
0       3894
7500      20
3500      15
3700      14
3880      13
        ... 
3227       1
3443       1
3433       1
2000       1
3052       1
Name: count, Length: 352, dtype: int64

In [31]:
engine_df['THRUST'] = engine_df['THRUST'].astype('Int64')
engine_df['THRUST'] = engine_df['THRUST'].replace(0, np.nan)

In [32]:
engine_df['THRUST'].value_counts(dropna=False)

THRUST
<NA>    3894
7500      20
3500      15
3700      14
3880      13
        ... 
3227       1
3443       1
3433       1
2000       1
3052       1
Name: count, Length: 352, dtype: Int64

In [35]:
engine_df

,CODE,MFR,MODEL,TYPE,HORSEPOWER,THRUST
0,0,NONE,NONE,None,<NA>,<NA>
1,401,A.C.E.,HIDR MARK III,Reciprocating,95,<NA>
2,402,A.C.E.,UPRI MARK III,Reciprocating,100,<NA>
3,450,AEROMOMENT,AM13 SERIES,4 Cycle,100,<NA>
4,452,AEROMOMENT,AM15 SERIES,4 Cycle,117,<NA>
...,...,...,...,...,...,...
4733,83358,KDE,7215XF,Electric,135,<NA>
4734,83359,KDE,HACKER,Electric,495,<NA>
4735,83360,KDE,10218XF-105,Electric,140,<NA>
4736,99222,MAGICALL,MAGIDRIVE 75,Electric,350,<NA>


## Column Name Cleaning

In [36]:
engine_df.columns = (engine_df.columns
                     .str.replace(' ', '_', regex=False)
                     .str.replace('-', '_', regex=False)
                     .str.replace('(', '', regex=False)
                     .str.replace(')', '', regex=False)
                     .str.lower())

In [40]:
engine_df.to_csv('./clean_data/ENGINE.csv', index=False, na_rep='NULL')